# **Data Preparation for Fine-Tuning - Medicines**

## **Data Preprocessing**  
**`"processed_medicines_vfinal.json"`**

### **1. Analysis of Unmapped Compositions**
Identifies medicines with unmapped active ingredients (denoted as `"(NA)"`) strictly within the `Composition` field.

In [1]:
import json
import os

INPUT_FILE = os.path.join("..", "..", "Data", "Pharmallm_Drug_Dataset_Uses_Side_Effects_and_User_Reviews", "processed_medicines_vfinal.json")

def count_na_in_composition_only():
    print(f"--- [START] Strict Analysis of '(NA)' in 'Composition' ---")

    if not os.path.exists(INPUT_FILE):
        print(f"[ERROR] File '{INPUT_FILE}' not found.")
        return

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)

        na_count = 0
        na_examples = []

        print("Scanning strictly the 'Composition' field for '(NA)'...")
        for item in data:
            composition_data = item.get('Composition', item.get('composition', ''))
            
            composition_str = str(composition_data)
            
            if "(NA)" in composition_str:
                na_count += 1
                
                if len(na_examples) < 10:
                    med_name = item.get('Medicine Name', item.get('medicine_name', 'Unknown Medicine'))
                    na_examples.append(f"[{med_name}]  -->  {composition_data}")

        total_records = len(data)
        percentage = (na_count / total_records) * 100 if total_records > 0 else 0

        print("\n" + "=" * 70)
        print(" STRICT COMPOSITION ANALYSIS SUMMARY")
        print("=" * 70)
        print(f" Total Medicines Scanned : {total_records}")
        print(f" Medicines with '(NA)' in Composition: {na_count} ({percentage:.2f}%)")
        print("-" * 70)
        
        if na_count > 0:
            print("\nPreview of medicines with '(NA)' strictly in Composition:")
            for example in na_examples:
                print(f" -> {example}")
            
            if na_count > 10:
                print(f" -> ... and {na_count - 10} more.")
        
        else:
            print("\nNo '(NA)' tags were found in the Composition field.")

    except json.JSONDecodeError:
        print("[ERROR] The file is not a valid JSON.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")

count_na_in_composition_only()

--- [START] Strict Analysis of '(NA)' in 'Composition' ---
Scanning strictly the 'Composition' field for '(NA)'...

 STRICT COMPOSITION ANALYSIS SUMMARY
 Total Medicines Scanned : 11825
 Medicines with '(NA)' in Composition: 77 (0.65%)
----------------------------------------------------------------------

Preview of medicines with '(NA)' strictly in Composition:
 -> [Alfakit Tablet]  -->  ['Alpha Ketoanalogue (NA)']
 -> [Asthakind-LS Drop]  -->  ['Ambroxol (NA)', 'Levosalbutamol (NA)', 'Guaifenesin (NA)']
 -> [Alfalog Tablet]  -->  ['Alpha Ketoanalogue (NA)']
 -> [Amcrylate Bio-Adhesive]  -->  ['Iso Amyl 2-Cynoacrylate (NA)']
 -> [Biofil-AB Particles]  -->  ['Mupirocin (2% w/w)', 'Metronidazole (1% w/w)', 'Collagen Peptide (NA)']
 -> [Biochemdryl Syrup]  -->  ['Menthol (NA)', 'Diphenhydramine (NA)', 'Ammonium Chloride (NA)']
 -> [DMR Syrup]  -->  ['Dextromethorphan Hydrobromide (NA)']
 -> [Diplomax Lotion]  -->  ['Clobetasol (NA)', 'Salicylic Acid (NA)']
 -> [Denon Soap]  -->  ['Ketoc

#### **1.1. Removal of Unmapped Compositions**
Filters out any medication lacking a fully mapped composition (`"(NA)"`) to ensure the model trains only on complete pharmacological data. The cleaned data is saved as `processed_medicines_strict_composition.json`.

In [2]:
import json
import os

INPUT_FILE = input_file = os.path.join("..", "..", "Data", "Pharmallm_Drug_Dataset_Uses_Side_Effects_and_User_Reviews", "processed_medicines_vfinal.json")
output_dir = "intermediate_data"
output_filename = "processed_medicines_strict_composition.json"
output_path = os.path.join(output_dir, output_filename)

def filter_na_in_composition():
    print(f"--- [START] Executing Removal of '(NA)' Compositions ---")

    if not os.path.exists(INPUT_FILE):
        print(f"[ERROR] File '{INPUT_FILE}' not found.")
        return

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)

        valid_data = []
        removed_count = 0

        print("Filtering out entries with '(NA)' strictly in the 'Composition' field...")
        for item in data:
            composition_data = item.get('Composition', item.get('composition', ''))
            
            composition_str = str(composition_data)
            
            # Route the item based on the presence of "(NA)"
            if "(NA)" in composition_str:
                removed_count += 1
            else:
                valid_data.append(item)

        os.makedirs(output_dir, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f_out:
            json.dump(valid_data, f_out, indent=4, ensure_ascii=False)

        total_records = len(data)
        
        print("\n" + "=" * 60)
        print(" COMPOSITION FILTERING SUMMARY")
        print("=" * 60)
        print(f" Initial Medicines Scanned : {total_records}")
        print(f" Medicines Removed ('NA')  : {removed_count}")
        print(f" Valid Medicines Retained  : {len(valid_data)}")
        print("-" * 60)
        print(f" Clean dataset saved to    : {output_path}")
        print("=" * 60)

    except json.JSONDecodeError:
        print("[ERROR] The file is not a valid JSON.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")

filter_na_in_composition()

--- [START] Executing Removal of '(NA)' Compositions ---
Filtering out entries with '(NA)' strictly in the 'Composition' field...

 COMPOSITION FILTERING SUMMARY
 Initial Medicines Scanned : 11825
 Medicines Removed ('NA')  : 77
 Valid Medicines Retained  : 11748
------------------------------------------------------------
 Clean dataset saved to    : intermediate_data\processed_medicines_strict_composition.json


### **2. Content-Based Deduplication**
Removes exact duplicate entries. The cleaned dataset is saved into the `intermediate_data/` folder as `processed_medicines_unique.json`.

**Note:** In the initial dataset (csv) the rows weren't duplicate, because they had url for the image of the medicine, manufacturer and rating. For example in the original dataset the medicine "Avil Injection" appeared 2 times, but with differente urls and ratings. In this case, as I'm not considering urls and ratings, the medicine would be exactly the same. So one of them will be removed.

In [ ]:
import json
import os
import hashlib

input_file = "intermediate_data/processed_medicines_strict_composition.json"
output_dir = "intermediate_data"
output_filename = "processed_medicines_unique.json"
output_path = os.path.join(output_dir, output_filename)

if os.path.exists(input_file):
    print(f"Loading raw data: {input_file}...")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    initial_count = len(data)
    print(f"Total Records Before: {initial_count}")

    # DEDUPLICATION
    # use of a set to store unique content hashes
    seen_hashes = set()
    unique_data = []
    
    duplicates_count = 0

    print("Scanning for duplicates...")

    for item in data:
        # Create a string representation of the dictionary
        # This acts as a "fingerprint". If the content is identical, the string is identical.
        item_str = json.dumps(item, sort_keys=True)
        
        # Generate a hash (MD5) for faster comparison than long strings
        item_hash = hashlib.md5(item_str.encode('utf-8')).hexdigest()
        
        if item_hash not in seen_hashes:
            seen_hashes.add(item_hash)
            unique_data.append(item)
        else:
            #if duplicates_count == 0:
            #    print("\n EXAMPLE OF DUPLICATE REMOVED (First Occurrence):")
            #    print(json.dumps(item, indent=4, ensure_ascii=False))
            #    print("-" * 50 + "\n")
            
            duplicates_count += 1

    print(f"Duplicates Removed: {duplicates_count}")
    print(f"Total Unique Records: {len(unique_data)}")
    
    if duplicates_count > 0:
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"Saving unique dataset to: {output_path}...")
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(unique_data, f, indent=4, ensure_ascii=False)
        print("Deduplication complete.")
    else:
        print("No duplicates found. The dataset is already clean.")

else:
    print(f"ERROR: Input file not found: {input_file}")

Loading raw data: intermediate_data/processed_medicines_strict_composition.json...
Total Records Before: 11748
Scanning for duplicates...
Duplicates Removed: 321
Total Unique Records: 11427
Saving unique dataset to: intermediate_data\processed_medicines_unique.json...
Deduplication complete.


### **3. Diagnostic: Verification of Unique Medicine Names**
Checks if any duplicate medicine names survived the content-based deduplication due to minor string formatting variations in other fields.

In [23]:
import json
import os
from collections import Counter

input_file = "intermediate_data/processed_medicines_unique.json"

if os.path.exists(input_file):
    print(f"Scanning file: {input_file}...")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"Total Records: {len(data)}")

    # Extract all names
    all_names = [item.get('Medicine Name') for item in data if item.get('Medicine Name')]
    
    # Count occurrences
    name_counts = Counter(all_names)
    
    # Filter duplicates 
    duplicates = {name: count for name, count in name_counts.items() if count > 1}
    
    print("\n--- DUPLICATES: ---")
    
    if duplicates:
        print(f"Found {len(duplicates)} medicine names that appear multiple times.")
        print("-" * 50)
        
        for i, (name, count) in enumerate(duplicates.items()):
            print(f" [{count}x] {name}")
            if i >= 19:
                print(f"... and {len(duplicates) - 20} more.")
                break
        print("-" * 50)
        print("CONCLUSION: Further deduplication by 'Medicine Name' is required.")
    else:
        print("No duplicate medicine names found. The dataset is strictly unique.")

else:
    print(f"ERROR: File not found: {input_file}")

Scanning file: intermediate_data/processed_medicines_unique.json...
Total Records: 11427

--- DUPLICATES: ---
Found 3 medicine names that appear multiple times.
--------------------------------------------------
 [2x] Basalog 100IU/ml Injection
 [2x] Gestone Injection
 [2x] Tegliptin Tablet
--------------------------------------------------
CONCLUSION: Further deduplication by 'Medicine Name' is required.


#### **3.1.Resolving "Basalog" Ambiguity**
A manual inspection of the dataset revealed a specific duplicate case for "Basalog 100IU/ml Injection". The dataset contains two entries for this medicine: one specifying the composition as "Insulin Glargine (100IU)" and another, more accurate entry, specifying "Insulin Glargine (100IU/ml)". This script removes the incomplete composition entry for "Basalog 100IU/ml Injection" while keeping the accurate one, updating the dataset in place.

In [24]:
import json
import os

file_path = "intermediate_data/processed_medicines_unique.json"

if os.path.exists(file_path):
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_before = len(data)
    print(f"Total Records Before: {total_before}")

    clean_data = []
    removed_count = 0

    for item in data:
        name = item.get('Medicine Name')
        composition = item.get('Composition', [])

        # Specific check for Basalog
        if name == "Basalog 100IU/ml Injection":
            # If it has the incomplete composition "100IU", remove it
            if "Insulin Glargine (100IU)" in composition:
                print(f" -> Removing incorrect entry: {name}")
                removed_count += 1
                continue 

        clean_data.append(item)

    if removed_count > 0:
        print(f"\nRecords Removed: {removed_count}")
        print(f"Total Records After: {len(clean_data)}")
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(clean_data, f, indent=4, ensure_ascii=False)
        
        print(f"Successfully updated: {file_path}")
    else:
        print("No changes made. The target entry was not found.")

else:
    print(f"ERROR: File '{file_path}' not found.")

Total Records Before: 11427
 -> Removing incorrect entry: Basalog 100IU/ml Injection

Records Removed: 1
Total Records After: 11426
Successfully updated: intermediate_data/processed_medicines_unique.json


#### **3.2. Targeted Smart Merge: "Tegliptin Tablet"**
Merges two entries for "Tegliptin Tablet" that have fragmented side effect profiles. The side effects are combined into a single, comprehensive list to prevent clinical data loss.

In [25]:
import json
import os

file_path = "intermediate_data/processed_medicines_unique.json"
TARGET_MEDICINE = "Tegliptin Tablet"

def merge_tegliptin_duplicates():
    print(f"--- [START] Targeted Merge for '{TARGET_MEDICINE}' ---")

    if not os.path.exists(file_path):
        print(f"[ERROR] File '{file_path}' not found.")
        return

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        valid_data = []
        target_entries = []
        merge_count = 0

        print(f"Scanning dataset for all instances of '{TARGET_MEDICINE}'...")
        
        for item in data:
            med_name = item.get('Medicine Name', '')
            if med_name == TARGET_MEDICINE:
                target_entries.append(item)
            else:
                valid_data.append(item)

        # Check if it has multiple entries to merge
        if len(target_entries) > 1:
            print(f"Found {len(target_entries)} conflicting entries for {TARGET_MEDICINE}.")
            
            # Initialize a set for side effects to automatically drop exact duplicates
            merged_side_effects = set()
            
            # Composition, Uses, and Mechanism are identical based on the analysis
            # Using the base structure of the first entry
            unified_entry = target_entries[0].copy()
            
            # Extract and merge all side effects from all target entries
            for entry in target_entries:
                effects = entry.get('Side_effects', [])
                for effect in effects:
                    # Strip spaces and add to set
                    merged_side_effects.add(effect.strip())
            
            unified_entry['Side_effects'] = sorted(list(merged_side_effects))
            
            # Add the newly merged entry to the clean dataset
            valid_data.append(unified_entry)
            
            merge_count += 1
            
            print(f"\nSuccessfully unified {len(target_entries)} entries into 1.")
            print("Unified Side Effects Profile:")
            for effect in unified_entry['Side_effects']:
                print(f" -> {effect}")
                
        elif len(target_entries) == 1:
            print(f"Only 1 entry found for {TARGET_MEDICINE}. No merge required.")
            valid_data.append(target_entries[0])
            
        else:
            print(f"No entries found for {TARGET_MEDICINE} in the current dataset.")

        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'w', encoding='utf-8') as f_out:
            json.dump(valid_data, f_out, indent=4, ensure_ascii=False)

        print("\n" + "=" * 60)
        print(" TARGETED MERGE SUMMARY")
        print("=" * 60)
        print(f" Target Medication           : {TARGET_MEDICINE}")
        print(f" Successful Merge Operations : {merge_count}")
        print(f" Total Medicines Retained    : {len(valid_data)}")
        print("-" * 60)
        print(f" Updated dataset saved to    : {file_path}")
        print("=" * 60)

    except json.JSONDecodeError:
        print("[ERROR] The file is not a valid JSON.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")

merge_tegliptin_duplicates()

--- [START] Targeted Merge for 'Tegliptin Tablet' ---
Scanning dataset for all instances of 'Tegliptin Tablet'...
Found 2 conflicting entries for Tegliptin Tablet.

Successfully unified 2 entries into 1.
Unified Side Effects Profile:
 -> Constipation
 -> Diarrhea
 -> Dizziness
 -> Headache
 -> Hypoglycemia low blood glucose level
 -> Increased body temperature
 -> Nausea
 -> Vomiting

 TARGETED MERGE SUMMARY
 Target Medication           : Tegliptin Tablet
 Successful Merge Operations : 1
 Total Medicines Retained    : 11425
------------------------------------------------------------
 Updated dataset saved to    : intermediate_data/processed_medicines_unique.json


#### **3.3. Resolving Packaging Variations: "Gestone Injection"**

There are two distinct entries for **"Gestone Injection"**. A review revealed that the difference between the two entries pertained strictly to the ampoule volume: one formulation was 50mg/1ml, while the other was 100mg/2ml (the dataset only has 100mg). 

This script specifically targets and removes the 100mg packaging variant of "Gestone Injection". The cleaned dataset is then saved directly over the original file.

In [26]:
import json
import os

FILE_PATH = "intermediate_data/processed_medicines_unique.json"

# target to remove
TARGET_MEDICINE = "Gestone Injection"
TARGET_COMPOSITION_SUBSTRING = "100mg"

def remove_redundant_gestone():
    print(f"--- [START] Removing Clinical Redundancy in '{FILE_PATH}' ---")

    if not os.path.exists(FILE_PATH):
        print(f"[ERROR] File '{FILE_PATH}' not found.")
        return

    try:
        with open(FILE_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)

        initial_count = len(data)
        valid_data = []
        removed_entries = []

        print(f"Scanning {initial_count} records for redundant '{TARGET_MEDICINE}' variants...")

        for item in data:
            med_name = item.get('Medicine Name', '')
            composition_str = str(item.get('Composition', ''))
            
            # If it's Gestone Injection AND contains 100mg in its composition, remove it
            if med_name == TARGET_MEDICINE and TARGET_COMPOSITION_SUBSTRING in composition_str:
                removed_entries.append(f"[{med_name}] --> Composition: {composition_str}")
            else:
                # Keep all other medicines
                valid_data.append(item)

        removed_count = len(removed_entries)

        with open(FILE_PATH, 'w', encoding='utf-8') as f_out:
            json.dump(valid_data, f_out, indent=4, ensure_ascii=False)

        print("\n" + "=" * 65)
        print(" REDUNDANCY REMOVAL SUMMARY")
        print("=" * 65)
        print(f" Initial Medicines       : {initial_count}")
        print(f" Records Removed         : {removed_count}")
        print(f" Final Medicines Retained: {len(valid_data)}")
        print("-" * 65)
        
        if removed_count > 0:
            print("\nSuccessfully removed the following redundant entries:")
            for entry in removed_entries:
                print(f" -> {entry}")
        else:
            print(f"\nNo entries matching '{TARGET_MEDICINE}' with '{TARGET_COMPOSITION_SUBSTRING}' were found.")
            
        print("-" * 65)
        print(f" File successfully updated in place: {FILE_PATH}")

    except json.JSONDecodeError:
        print("[ERROR] The file is not a valid JSON.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")

remove_redundant_gestone()

--- [START] Removing Clinical Redundancy in 'intermediate_data/processed_medicines_unique.json' ---
Scanning 11425 records for redundant 'Gestone Injection' variants...

 REDUNDANCY REMOVAL SUMMARY
 Initial Medicines       : 11425
 Records Removed         : 1
 Final Medicines Retained: 11424
-----------------------------------------------------------------

Successfully removed the following redundant entries:
 -> [Gestone Injection] --> Composition: ['Progesterone (100mg)']
-----------------------------------------------------------------
 File successfully updated in place: intermediate_data/processed_medicines_unique.json


---

## **Data Split**

#### **`processed_medicines_unique.json`**

### **1. Splitting the Dataset - v1 (Random Split)**
Performs a standard random split (70% Train, 10% Dev, 20% Test). 

In [27]:
import json
import random
import os


input_file = "intermediate_data/processed_medicines_unique.json"

output_dir = "datasets_split"
output_train = os.path.join(output_dir, "dataset_meds_TRAIN_v1.json")
output_dev   = os.path.join(output_dir, "dataset_meds_DEV_v1.json")
output_test  = os.path.join(output_dir, "dataset_meds_TEST_v1.json")

# Split Ratios
TRAIN_RATIO = 0.70
DEV_RATIO   = 0.10
TEST_RATIO  = 0.20

if os.path.exists(input_file):
    print(f"Loading: {input_file}...")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_meds = len(data)
    print(f"Total Medicines: {total_meds}")
    os.makedirs(output_dir, exist_ok=True)

    random.seed(42)
    random.shuffle(data)

    idx_train = int(total_meds * TRAIN_RATIO)
    idx_dev   = int(total_meds * (TRAIN_RATIO + DEV_RATIO))

    train_data = data[:idx_train]
    dev_data   = data[idx_train:idx_dev]
    test_data  = data[idx_dev:]


    with open(output_train, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_train}: {len(train_data)} records (70%)")

    with open(output_dev, 'w', encoding='utf-8') as f:
        json.dump(dev_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_dev}: {len(dev_data)} records (10%)")

    with open(output_test, 'w', encoding='utf-8') as f:
        json.dump(test_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_test}: {len(test_data)} records (20%)")

    print("\nData splitting completed successfully. Files saved")

else:
    print(f"ERROR: File '{input_file}' not found.")

Loading: intermediate_data/processed_medicines_unique.json...
Total Medicines: 11424
 -> datasets_split\dataset_meds_TRAIN_v1.json: 7996 records (70%)
 -> datasets_split\dataset_meds_DEV_v1.json: 1143 records (10%)
 -> datasets_split\dataset_meds_TEST_v1.json: 2285 records (20%)

Data splitting completed successfully. Files saved


### **2. Splitting the Dataset - v2 (Advanced Grouped Splitting)**
Groups medications using a composite key:  **[First Word of Medicine Name] + [Base Composition]**. This tries to ensure that all dosages and variants of the exact same commercial drug fall strictly into the same split (Train, Dev, or Test)

In [ ]:
import json
import random
import os
import re
from collections import defaultdict

input_file = "intermediate_data/processed_medicines_unique.json"

output_dir = "datasets_split"
output_train = os.path.join(output_dir, "dataset_meds_TRAIN_v2.json")
output_dev   = os.path.join(output_dir, "dataset_meds_DEV_v2.json")
output_test  = os.path.join(output_dir, "dataset_meds_TEST_v2.json")

TRAIN_RATIO = 0.70
DEV_RATIO   = 0.10
TEST_RATIO  = 0.20

def get_base_composition(composition_list):
    """
    Strips dosages from the composition list to extract pure active ingredients.
    Example: 'Cefixime (100mg)' -> 'Cefixime'
    """
    base_ingredients = []
    for ingredient in composition_list:
        clean_ing = re.sub(r'\(.*?\)', '', ingredient).strip()
        base_ingredients.append(clean_ing)
    
    return tuple(sorted(base_ingredients))

if os.path.exists(input_file):
    print(f"--- [START] Advanced Grouped Splitting ---")
    print(f"Loading: {input_file}...")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_meds = len(data)
    print(f"Total Medicine Records: {total_meds}")
    os.makedirs(output_dir, exist_ok=True)

    # Group by (First Word, Base Composition)
    grouped_data = defaultdict(list)
    
    for item in data:
        med_name = item.get("Medicine Name", "Unknown").strip()
        composition = item.get("Composition", [])
        
        if isinstance(composition, str):
            composition = [composition]
            
        first_word = med_name.split()[0] if med_name else "Unknown"
        base_comp = get_base_composition(composition)
        
        # Create the composite key
        group_key = (first_word, base_comp)
        grouped_data[group_key].append(item)
        
    unique_groups = list(grouped_data.keys())
    print(f"Total Unique Pharmacological Groups: {len(unique_groups)}")

    # Shuffle the groups
    random.seed(42)
    random.shuffle(unique_groups)

    # Calculate target sizes
    target_train = int(total_meds * TRAIN_RATIO)
    target_dev   = int(total_meds * DEV_RATIO)

    train_data = []
    dev_data   = []
    test_data  = []

    # Allocate groups
    print("Allocating grouped data into Train, Dev, and Test sets...")
    for key in unique_groups:
        group = grouped_data[key]
        
        if len(train_data) < target_train:
            train_data.extend(group)
        elif len(dev_data) < target_dev:
            dev_data.extend(group)
        else:
            test_data.extend(group)

    with open(output_train, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_train}: {len(train_data)} records ({(len(train_data)/total_meds)*100:.2f}%)")

    with open(output_dev, 'w', encoding='utf-8') as f:
        json.dump(dev_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_dev}: {len(dev_data)} records ({(len(dev_data)/total_meds)*100:.2f}%)")

    with open(output_test, 'w', encoding='utf-8') as f:
        json.dump(test_data, f, indent=4, ensure_ascii=False)
    print(f" -> {output_test}: {len(test_data)} records ({(len(test_data)/total_meds)*100:.2f}%)")

    print("\n[SUCCESS] Advanced Grouped Data Splitting completed perfectly.")

else:
    print(f"[ERROR] File '{input_file}' not found.")

--- [START] Advanced Grouped Splitting ---
Loading: intermediate_data/processed_medicines_unique.json...
Total Medicine Records: 11424
Total Unique Pharmacological Groups: 8028
Allocating grouped data into Train, Dev, and Test sets...
 -> datasets_split\dataset_meds_TRAIN_v2.json: 7997 records (70.00%)
 -> datasets_split\dataset_meds_DEV_v2.json: 1154 records (10.10%)
 -> datasets_split\dataset_meds_TEST_v2.json: 2273 records (19.90%)

[SUCCESS] Advanced Grouped Data Splitting completed perfectly.


### **3. Data Split Verification**
Checks for any "Medicine Name" overlaps across the Train, Dev, and Test sets.

In [29]:
import json
import os

INPUT_DIR = "datasets_split"

files = {
    "train": os.path.join(INPUT_DIR, "dataset_meds_TRAIN_v2.json"),
    "dev":   os.path.join(INPUT_DIR, "dataset_meds_DEV_v2.json"),
    "test":  os.path.join(INPUT_DIR, "dataset_meds_TEST_v2.json")
}


def load_names(filepath):
    """Loads a JSON file and extracts a set of Medicine Names."""
    if not os.path.exists(filepath):
        print(f"WARNING: File not found: {filepath}")
        return set()
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # Extract names
        names = {item.get('Medicine Name') for item in data if item.get('Medicine Name')}
    return names

print("--- Verifying Data Integrity ---")

# Load Sets
train_names = load_names(files["train"])
dev_names   = load_names(files["dev"])
test_names  = load_names(files["test"])

print(f"Loaded unique medicines:")
print(f" -> Train: {len(train_names)}")
print(f" -> Dev:   {len(dev_names)}")
print(f" -> Test:  {len(test_names)}")

# Check Intersections (Overlaps)
overlap_train_dev = train_names.intersection(dev_names)
overlap_train_test = train_names.intersection(test_names)
overlap_dev_test   = dev_names.intersection(test_names)

print("\n--- Overlap Report ---")

has_error = False

if overlap_train_dev:
    print(f"[CRITICAL] Found {len(overlap_train_dev)} overlaps between TRAIN and DEV!")
    print(f"Examples: {list(overlap_train_dev)[:5]}")
    has_error = True
else:
    print("[OK] No overlap between TRAIN and DEV.")

if overlap_train_test:
    print(f"[CRITICAL] Found {len(overlap_train_test)} overlaps between TRAIN and TEST!")
    print(f"Examples: {list(overlap_train_test)[:5]}")
    has_error = True
else:
    print("[OK] No overlap between TRAIN and TEST.")

if overlap_dev_test:
    print(f"[CRITICAL] Found {len(overlap_dev_test)} overlaps between DEV and TEST!")
    print(f"Examples: {list(overlap_dev_test)[:5]}")
    has_error = True
else:
    print("[OK] No overlap between DEV and TEST.")

print("-" * 30)
if not has_error:
    print("SUCCESS: The datasets are completely disjoint (No Data Leakage).")
else:
    print("FAILURE: Data leakage detected. Check the splitting logic.")

--- Verifying Data Integrity ---
Loaded unique medicines:
 -> Train: 7997
 -> Dev:   1154
 -> Test:  2273

--- Overlap Report ---
[OK] No overlap between TRAIN and DEV.
[OK] No overlap between TRAIN and TEST.
[OK] No overlap between DEV and TEST.
------------------------------
SUCCESS: The datasets are completely disjoint (No Data Leakage).


---

## **QA Generation** 

**`processed_medicines_unique.json`**

### **1. Creation of the instruction-tuning dataset for Training and Validation**

Transforms the pharmacological database into Q&A pairs across 4 clinical evaluation pillars:
1.  **Pillar 1:** presents a brand-name medicine and requests its full clinical profile, including composition, indications, and side effects. In instances where no side effects were identified after data cleaning, a standardized placeholder (”Unknown safety profile.”) was used in the resultant output.
2. **Pillar 2:** presents a clinical condition and requests the active substances indicated for its treatment. Reverses the search logic. It groups the database by clinical indications, teaching the model to recommend active principles for a given disease. It appends a mandatory clinical disclaimer regarding individualized dosage and formulation selection.
3. **Pillar 3:** presents an active-ingredient composition and requests the matching commercial brands.
4. **Pillar 4:** presents an active substance and requests its MoA. Extracts and strictly deduplicates the MoA by active ingredient.

In [30]:
import json
import random
import os
import re

random.seed(42)

INPUT_DIR = "datasets_split"      
OUTPUT_DIR = "data_finetuning"    

FILES = {
    "train_in": os.path.join(INPUT_DIR, "dataset_meds_TRAIN_v2.json"), 
    "dev_in":   os.path.join(INPUT_DIR, "dataset_meds_DEV_v2.json"),
    
    "train_out": os.path.join(OUTPUT_DIR, "Finetuning_medicines_TRAIN.json"),
    "dev_out":   os.path.join(OUTPUT_DIR, "Finetuning_medicines_DEV.json")
}

INSTRUCTIONS_PILLAR_1 = [
    "You are a clinical pharmacology assistant supporting a licensed physician. Provide the complete clinical profile and safety data for the requested brand-name medicine to aid in prescribing decisions.",
    "As a clinical decision support system, deliver accurate pharmacological information directly to the treating physician. Include composition, indications, and adverse effects for the specified brand-name medicine.",
    "Assist the physician with drug information. Generate a comprehensive clinical summary for the requested brand-name medicine, including active ingredients, approved indications, and documented side effects."
]

INSTRUCTIONS_PILLAR_2 = [
    "You are assisting a physician with treatment selection. Identify pharmacological options indicated for the specified medical condition to support clinical decision-making.",
    "As a clinical pharmacology consultant, provide the physician with available drug treatments for the patient's condition. List compositions indicated for the given diagnosis.",
    "Assist the physician in therapeutic planning. Identify the pharmacological compositions and active ingredients clinically indicated for the specified medical condition."
]

INSTRUCTIONS_PILLAR_3 = [
    "You are assisting a physician with drug availability. Identify which brand-name medicines contain the specified active ingredient composition to support prescribing.",
    "Provide the treating physician with brand-name options for the requested active ingredient(s). This helps identify available formulations.",
    "As a clinical pharmacology assistant, match the specified active ingredient composition to its corresponding commercial brand-name products to assist the physician."
]

INSTRUCTIONS_PILLAR_4 = [
    "Explain the mechanism of action of the specified active ingredient to support the physician's understanding of how the drug produces its therapeutic effect.",
    "You are a clinical pharmacology assistant. Assist the physician by detailing the mechanism of action of the requested active ingredient to support clinical decision-making."
]


PROMPTS_PILLAR_1 = [
    "Provide the clinical profile, including composition, indications, and side effects, for {med_name}.",
    "What are the clinical indications, composition, and side effects of {med_name}?"
]

PROMPTS_PILLAR_2 = [
    "Which drugs are indicated for {condition}?",
    "List the drugs indicated for {condition}."
]

PROMPTS_PILLAR_3 = [
    "List the brand names available for medicines containing {composition}.",
    "Which brand-name medicines contain {composition}?"
]

PROMPTS_PILLAR_4 = [
    "What is the mechanism of action of {ingredient}?",
    "Explain the mechanism of action of {ingredient}."
]

# --- HELPERS ---
def natural_join(lst):
    if not lst: return ""
    if len(lst) == 1: return lst[0]
    return ", ".join(lst[:-1]) + " and " + lst[-1]

def clean_composition_text(text):
    # Strip dosages and make lowercase
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

def analyze_side_effects(effects_list):
    if not effects_list: return None, False
    cleaned_list = []
    found_safe_phrase = False
    
    for effect in effects_list:
        if not effect: continue 
        if "No common side effects seen" in effect:
            found_safe_phrase = True
            continue 
        clean_effect = effect.strip(" .") 
        if clean_effect: 
            cleaned_list.append(clean_effect)
            
    if found_safe_phrase and not cleaned_list:
        return None, True 
    if not cleaned_list:
        return None, False
    return cleaned_list, False

# --- MAIN FUNCTION ---
def generate_cdss_dataset(data):
    dataset = []
    
    counters = {
        "pillar1_profiles": 0,
        "pillar2_reasoning": 0,
        "pillar3_inventory": 0,
        "pillar4_moa": 0
    }
    
    # Dictionaries to build Pillars 2, 3, and 4
    use_to_compositions = {} 
    comp_to_brands = {}      
    unique_moa_registry = {} 

    # ==========================================
    # DATA EXTRACTION & PILLAR 1 (Brand -> Profile)
    # ==========================================

    for item in data:
        name = item.get('Medicine Name')
        uses = [u.strip() for u in item.get('Uses', []) if u.strip()]
        raw_side_effects = item.get('Side_effects', [])
        comp_raw_list = item.get('Composition', [])
        moa_dict = item.get('Mechanism_of_action', {})

        if not name or not comp_raw_list: continue

        # Display Composition (Keeps dosages for Pillar 1)
        comp_display_str = natural_join([c.strip() for c in comp_raw_list])

        # Clean Composition (Strips dosages for Pillars 2 and 3 Grouping)
        comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
        comp_str = natural_join(comp_clean_list)

        # Populate Index for Pilar 2 (Uses)
        for use in uses:
            use_lower = use.lower()
            if use_lower not in use_to_compositions:
                use_to_compositions[use_lower] = set()
            use_to_compositions[use_lower].add(comp_str)

        # Populate Index for Pilar 3 (Brands)
        if comp_str not in comp_to_brands:
            comp_to_brands[comp_str] = set()
        comp_to_brands[comp_str].add(name)

        # Populate Index for Pilar 4 (MoA) - Ensuring uniqueness
        if moa_dict:
            for ing, text in moa_dict.items():
                if text and text.strip():
                    ing_clean = ing.strip().title()
                    if ing_clean not in unique_moa_registry:
                        unique_moa_registry[ing_clean] = text.strip()

        # --- GENERATE PILLAR 1 QA ---
        side_effects, is_explicitly_safe = analyze_side_effects(raw_side_effects)
        
        # Build Output with Bullets
        ans_lines = [
            f"**Medicine: {name}:**",
            f"- **Composition:** {comp_display_str}",
            f"- **Indications:**"
        ]
        
        # Add Indications as bullets
        for u in uses:
            ans_lines.append(f"  - {u}")
            
        ans_lines.append(f"- **Side Effects:**")
        
        # Add Side Effects as bullets
        if is_explicitly_safe:
            ans_lines.append("  - No common side effects reported.")
        elif side_effects:
            for se in side_effects:
                ans_lines.append(f"  - {se}")
        else:
            ans_lines.append("  - Unknown safety profile.")

        dataset.append({
            "instruction": random.choice(INSTRUCTIONS_PILLAR_1),
            "input": random.choice(PROMPTS_PILLAR_1).format(med_name=name),
            "output": "\n".join(ans_lines)
        })
        counters["pillar1_profiles"] += 1

    # ==========================================
    # PILLAR 2: THERAPEUTIC REASONING (Disease -> Composition)
    # ==========================================

    for use, comps_set in use_to_compositions.items():
        comps_list = sorted(list(comps_set))
        
        comps_bullets = []
        for c in comps_list:
            comp_clean = c.title().replace(" And ", " and ")
            comps_bullets.append(f"- {comp_clean}")
            
        comps_formatted = "\n".join(comps_bullets)
        
        output_text = (
            f"The drug(s) indicated for {use} include:\n{comps_formatted}\n\n"
            f"*Clinical note: The selection of a specific drug, dosage, and formulation must be individualized based on patient-specific clinical factors, contraindications, and potential drug–drug interactions.*"
        )
        
        dataset.append({
            "instruction": random.choice(INSTRUCTIONS_PILLAR_2),
            "input": random.choice(PROMPTS_PILLAR_2).format(condition=use),
            "output": output_text
        })
        counters["pillar2_reasoning"] += 1

    # ==========================================
    # PILLAR 3: Medicines knowledge (Composition -> Brand)
    # ==========================================
    for comp_str, brands_set in comp_to_brands.items():
        brands_list = sorted(list(brands_set))
        
        # build the bullet points for the brand names
        brands_bullets = []
        for b in brands_list:
            brands_bullets.append(f"- {b}")
            
        brands_formatted = "\n".join(brands_bullets)
        
        # fix the "And" capitalization in the input and intro sentence
        comp_display = comp_str.title().replace(" And ", " and ")
        
        output_text = f"The following brand(s) contain {comp_display}:\n{brands_formatted}"
        
        dataset.append({
            "instruction": random.choice(INSTRUCTIONS_PILLAR_3),
            "input": random.choice(PROMPTS_PILLAR_3).format(composition=comp_display),
            "output": output_text
        })
        counters["pillar3_inventory"] += 1

    # ==========================================
    # PILLAR 4: Active Ingredient -> MoA
    # ==========================================
    for ingredient, moa_text in unique_moa_registry.items():
        output_text = f"**Mechanism of Action of {ingredient}:**\n{moa_text}"
        
        dataset.append({
            "instruction": random.choice(INSTRUCTIONS_PILLAR_4),
            "input": random.choice(PROMPTS_PILLAR_4).format(ingredient=ingredient),
            "output": output_text
        })
        counters["pillar4_moa"] += 1

    # Shuffle the final dataset
    random.shuffle(dataset)
    return dataset, counters

def process_file(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if os.path.exists(input_path):
        with open(input_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        
        qa_data, counters = generate_cdss_dataset(raw_data)
        
        print(f"\n--- Statistics for {os.path.basename(input_path)} ---")
        print(f"  Pillar 1 (Clinical Profiles):   {counters['pillar1_profiles']}")
        print(f"  Pillar 2 (Uses -> Substances):  {counters['pillar2_reasoning']}")
        print(f"  Pillar 3 (Substances -> Brand): {counters['pillar3_inventory']}")
        print(f"  Pillar 4 (Unique MoA rules):    {counters['pillar4_moa']}")
        print(f"  --------------------------------------")
        print(f"  -> TOTAL QA PAIRS GENERATED:    {len(qa_data)}")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(qa_data, f, indent=4, ensure_ascii=False)
        print(f"  -> Saved to: {output_path}")
    else:
        print(f"WARNING: File not found: {input_path}")

if __name__ == "__main__":
    print("--- Generating CDSS Medicines Dataset ---")
    process_file(FILES["train_in"], FILES["train_out"])
    process_file(FILES["dev_in"],   FILES["dev_out"])
    print("--- Completed ---")

--- Generating CDSS Medicines Dataset ---



--- Statistics for dataset_meds_TRAIN_v2.json ---
  Pillar 1 (Clinical Profiles):   7997
  Pillar 2 (Uses -> Substances):  598
  Pillar 3 (Substances -> Brand): 1382
  Pillar 4 (Unique MoA rules):    714
  --------------------------------------
  -> TOTAL QA PAIRS GENERATED:    10691
  -> Saved to: data_finetuning\Finetuning_medicines_TRAIN.json

--- Statistics for dataset_meds_DEV_v2.json ---
  Pillar 1 (Clinical Profiles):   1154
  Pillar 2 (Uses -> Substances):  312
  Pillar 3 (Substances -> Brand): 494
  Pillar 4 (Unique MoA rules):    353
  --------------------------------------
  -> TOTAL QA PAIRS GENERATED:    2313
  -> Saved to: data_finetuning\Finetuning_medicines_DEV.json
--- Completed ---


#### **1.1. Dataset Profiling**
Extracts unique counts for compositions, uses, and MoAs to evaluate the dataset's (`dataset_meds_TRAIN_v2.json` and `dataset_meds_DEV_v2.json`) clinical diversity before training.

In [31]:
import json
import os
import re

INPUT_DIR = "datasets_split"      
FILES = [
    os.path.join(INPUT_DIR, "dataset_meds_TRAIN_v2.json"),
    os.path.join(INPUT_DIR, "dataset_meds_DEV_v2.json")
]

# --- HELPERS ---
def natural_join(lst):
    """Joins a list of strings naturally with commas and 'and'."""
    if not lst: return ""
    if len(lst) == 1: return lst[0]
    return ", ".join(lst[:-1]) + " and " + lst[-1]

def clean_composition_text(text):
    """Strips dosage information (e.g., '(10mg)') and converts to lowercase for exact matching."""
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

# --- MAIN FUNCTION ---
def analyze_dataset(file_path):
    """Reads a JSON dataset and calculates the absolute unique entities for Compositions, Uses, and MoAs."""
    if not os.path.exists(file_path):
        print(f"[WARNING] File not found: {file_path}")
        return

    # Using sets to automatically discard duplicate values
    unique_uses = set()
    unique_compositions = set()
    unique_moas = set()

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        # Extract Unique Clinical Uses (Indications)
        uses_raw = item.get('Uses', [])
        for u in uses_raw:
            clean_use = u.strip()
            if clean_use:
                unique_uses.add(clean_use.lower()) # Lowercase 

        # Extract Unique Compositions (Base active ingredients without dosages)
        comp_raw_list = item.get('Composition', [])
        if comp_raw_list:
            comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
            comp_str = natural_join(comp_clean_list)
            unique_compositions.add(comp_str)

        # Extract Unique Mechanisms of Action (Counted by active ingredient)
        moa_dict = item.get('Mechanism_of_action', {})
        if moa_dict:
            for ing, text in moa_dict.items():
                if text and text.strip():
                    unique_moas.add(ing.strip().title())

    print(f"\n{'='*50}")
    print(f" DATASET STATISTICS: {os.path.basename(file_path)}")
    print(f"{'='*50}")
    print(f" -> Total Commercial Medicines (Brands): {len(data)}")
    print(f" -> Unique Compositions (Active Groups): {len(unique_compositions)}")
    print(f" -> Unique Clinical Uses (Indications):  {len(unique_uses)}")
    print(f" -> Unique Mechanisms of Action (Rules): {len(unique_moas)}")
    print(f"{'='*50}\n")

if __name__ == "__main__":
    print("Starting dataset profiling...")
    for file in FILES:
        analyze_dataset(file)
    print("Profiling complete.")

Starting dataset profiling...

 DATASET STATISTICS: dataset_meds_TRAIN_v2.json
 -> Total Commercial Medicines (Brands): 7997
 -> Unique Compositions (Active Groups): 1382
 -> Unique Clinical Uses (Indications):  598
 -> Unique Mechanisms of Action (Rules): 714


 DATASET STATISTICS: dataset_meds_DEV_v2.json
 -> Total Commercial Medicines (Brands): 1154
 -> Unique Compositions (Active Groups): 494
 -> Unique Clinical Uses (Indications):  312
 -> Unique Mechanisms of Action (Rules): 353

Profiling complete.


### **2. Creation of the instruction-tuning dataset for Test**

#### **2.1. Open QA Format**
Generates standard open-ended questions for evaluation

In [32]:
FILES = {
    "test_in": os.path.join(INPUT_DIR, "dataset_meds_TEST_v2.json"), 
    "test_out": os.path.join(OUTPUT_DIR, "Finetuning_medicines_TEST_open.json")
}

if __name__ == "__main__":
    print("--- Generating CDSS Medicines TEST Dataset ---")
    process_file(FILES["test_in"], FILES["test_out"])
    print("--- Completed ---")

--- Generating CDSS Medicines TEST Dataset ---

--- Statistics for dataset_meds_TEST_v2.json ---
  Pillar 1 (Clinical Profiles):   2273
  Pillar 2 (Uses -> Substances):  406
  Pillar 3 (Substances -> Brand): 747
  Pillar 4 (Unique MoA rules):    459
  --------------------------------------
  -> TOTAL QA PAIRS GENERATED:    3885
  -> Saved to: data_finetuning\Finetuning_medicines_TEST_open.json
--- Completed ---


##### **2.1.1. Dataset Profiling**

In [33]:
INPUT_DIR = "datasets_split"      
file = os.path.join(INPUT_DIR, "dataset_meds_TEST_v2.json")

analyze_dataset(file)


 DATASET STATISTICS: dataset_meds_TEST_v2.json
 -> Total Commercial Medicines (Brands): 2273
 -> Unique Compositions (Active Groups): 747
 -> Unique Clinical Uses (Indications):  406
 -> Unique Mechanisms of Action (Rules): 459



#### **2.2. Multiple Choice Questions (MCQ) Format**

Transforms the test data into an MCQ format to evaluate clinical accuracy across four distinct tasks. 

**Question Types (Evaluation Pillars):**
* **Composition Identification:** given a brand-name medicine, identify its exact active-ingredient composition
* **Therapeutic Indications:** given a clinical condition, select the correct indicated treatments.
* **Brand Name Formulation:** - given an active-ingredient composition, identify the matching
commercial brands.
* **Mechanism of Action:** Explaining the mechanism of action of an active substance.

**-Internal vs. External Connectors:** Uses `and` to connect active ingredients within a single formulation, and `;` to separate distinct therapeutic options or brands.

**-Correct Answer Masking:** * **≤ 3 items:** Joins all correct items using `;` , **> 3 items:** Randomly selects one correct item to prevent the model from guessing the answer based on the length.

**-Dynamic Distractor Sizing:** Wrong options are dynamically constructed by randomly joining 1 to 3 items, ensuring all choices look visually similar in length and structure.

**-Filtering:** Employs a bidirectional overlap check (`Disease in Use` OR `Use in Disease`) when generating incorrect options for the Indications pillar. This rigorously prevents the accidental inclusion of "correct distractors" (valid alternative treatments) in the wrong choices.

**-Zero-Leakage MoA Masking:** Blind-tests the model by dynamically replacing the drug's generic name and all known synonyms with "this drug" inside the text. It uses `", "` for splitting and longest-to-shortest term sorting to prevent partial word masking, ensuring the model relies on pure clinical reasoning rather than keyword matching.

In [ ]:
import json
import random
import os
import re

random.seed(42)

INPUT_DIR = "datasets_split"      
OUTPUT_DIR = "data_finetuning"    

INPUT_FILE = os.path.join(INPUT_DIR, "dataset_meds_TEST_v2.json")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "Finetuning_medicines_TEST_mcq.json")

INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- HELPERS ---
def join_components(lst):
    if not lst: return ""
    if len(lst) == 1: return lst[0]
    return ", ".join(lst[:-1]) + " and " + lst[-1]

def join_options(lst):
    if not lst: return ""
    return "; ".join(lst)

def clean_composition_text(text):
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

def mask_drug_names(text, generic_name, synonyms_string):
    """
    Replaces the generic name AND synonyms with 'this drug' in the text.
    Uses smart splitting to avoid breaking IUPAC chemical names.
    """
    if not text: 
        return text, False, False
        
    replaced_generic = False
    replaced_synonym = False
    
    # Prepare the list of terms to mask
    terms_to_mask = [generic_name.strip()]
    
    if synonyms_string:
        # Split by comma followed by a space
        raw_syns = [s.strip() for s in synonyms_string.split(', ')]
        terms_to_mask.extend(raw_syns) #add the synonims to the list
        
    # Remove duplicates and sort by length (longest to shortest)
    terms_to_mask = sorted(list(set([t for t in terms_to_mask if t])), key=len, reverse=True)
    
    masked_text = text
    for term in terms_to_mask:
        # Escape the term to safely handle parentheses and brackets in regex
        pattern = re.compile(rf'\b{re.escape(term)}\b', re.IGNORECASE)
        
        # Check if the term exists in the current text 
        if pattern.search(masked_text):
            if term.lower() == generic_name.lower():
                replaced_generic = True
            else:
                replaced_synonym = True
                
            def smart_replace(match):
                # If the original word starts with a capital letter, keep it capitalized
                if match.group()[0].isupper():
                    return "This drug"
                return "this drug"
                
            masked_text = pattern.sub(smart_replace, masked_text)
            
    return masked_text, replaced_generic, replaced_synonym

def build_simple_mcq(correct_answer, distractor_pool):
    valid_distractors = [d for d in distractor_pool if d != correct_answer]
    distractors = random.sample(valid_distractors, min(3, len(valid_distractors)))
    options = [correct_answer] + distractors
    random.shuffle(options)
    letters = ['A', 'B', 'C', 'D']
    correct_letter = letters[options.index(correct_answer)]
    options_text = "\n".join([f"{letters[i]}. {opt}" for i, opt in enumerate(options)])
    return options_text, correct_letter

def build_hybrid_mcq(correct_items_list, global_pool, current_use=None, item_to_uses=None):
    """
    Builds an MCQ with randomly sized options (1 to 3 items joined by ';').
    Includes a semantic filter to prevent distractors from sharing 
    similar clinical indications with the correct answer.
    """
    valid_distractors = []
    local_filtered_count = 0
    fallback_triggered = False
    
    for d in global_pool:
        if d in correct_items_list:
            continue
            
        # Bidirectional semantic overlap check
        if current_use and item_to_uses:
            distractor_uses = item_to_uses.get(d, set())
            current_lower = current_use.lower().strip()
            has_overlap = False
            
            for d_use in distractor_uses:
                d_use_lower = d_use.lower().strip()
                if (current_lower in d_use_lower) or (d_use_lower in current_lower):
                    has_overlap = True
                    break
                    
            if has_overlap:
                local_filtered_count += 1
                continue 
                
        valid_distractors.append(d)
        
    # Format the correct answer
    if len(correct_items_list) <= 3:
        correct_text = join_options(correct_items_list)
    else:
        correct_text = random.choice(correct_items_list)
        
    # Determine random lengths for the 3 distractor options
    distractor_lengths = [random.randint(1, 3) for _ in range(3)]
    total_needed = sum(distractor_lengths)
    
    # Sample the distractors
    if len(valid_distractors) >= total_needed:
        sampled_distractors = random.sample(valid_distractors, total_needed)
    else:
        fallback_triggered = True
        sampled_distractors = random.choices(valid_distractors, k=total_needed)

    # Construct the 3 wrong options
    distractor_options = []
    start_idx = 0
    for length in distractor_lengths:
        chunk = sampled_distractors[start_idx : start_idx + length]
        distractor_options.append(join_options(chunk))
        start_idx += length
        
    # Assemble and shuffle
    options = [correct_text] + distractor_options
    random.shuffle(options)
    
    letters = ['A', 'B', 'C', 'D']
    correct_letter = letters[options.index(correct_text)]
    options_text = "\n".join([f"{letters[i]}. {opt}" for i, opt in enumerate(options)])
    
    return options_text, correct_letter, local_filtered_count, fallback_triggered

# --- MAIN FUNCTION ---
def generate_mcq_dataset(data, synonyms_dict):
    dataset_mcq = []
    pool_compositions = set()
    pool_brands = set()
    use_to_compositions = {}
    comp_to_brands = {}
    unique_moa_registry = {}
    
    comp_to_all_uses = {} 
    
    total_semantic_filtered = 0
    total_fallbacks = 0
    
    stats_masked_generic = 0
    stats_masked_synonym = 0
    stats_unique_texts_masked = 0 #counter for exact overlap-free masking

    for item in data:
        name = item.get('Medicine Name')
        uses = [u.strip() for u in item.get('Uses', []) if u.strip()]
        comp_raw_list = item.get('Composition', [])
        moa_dict = item.get('Mechanism_of_action', {})

        if not name or not comp_raw_list: continue

        comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
        comp_str = join_components(comp_clean_list).title().replace(" And ", " and ")
        
        pool_compositions.add(comp_str)
        pool_brands.add(name)
        
        if comp_str not in comp_to_all_uses:
            comp_to_all_uses[comp_str] = set()

        for use in uses:
            use_lower = use.lower()
            if use_lower not in use_to_compositions:
                use_to_compositions[use_lower] = set()
            use_to_compositions[use_lower].add(comp_str)
            comp_to_all_uses[comp_str].add(use_lower)

        if comp_str not in comp_to_brands:
            comp_to_brands[comp_str] = set()
        comp_to_brands[comp_str].add(name)

        if moa_dict:
            for ing, text in moa_dict.items():
                if text and text.strip():
                    ing_clean = ing.strip().title()
                    if ing_clean not in unique_moa_registry:
                        unique_moa_registry[ing_clean] = text.strip()

    list_all_compositions = list(pool_compositions)
    list_all_brands = list(pool_brands)

    # --- Pre-mask all MoA ---
    masked_moa_registry = {}
    
    for ingredient, moa_text in unique_moa_registry.items():
        syns_string = synonyms_dict.get(ingredient, "")
        masked_moa, used_gen, used_syn = mask_drug_names(moa_text, ingredient, syns_string)
        
        masked_moa_registry[ingredient] = masked_moa
        
        if used_gen: stats_masked_generic += 1
        if used_syn: stats_masked_synonym += 1
        if masked_moa != moa_text:
            stats_unique_texts_masked += 1
    
    list_all_moas = list(masked_moa_registry.values())

    # --- PILLAR 1: Composition Identification ---
    for item in data:
        name = item.get('Medicine Name')
        comp_raw_list = item.get('Composition', [])
        if not name or not comp_raw_list: continue
        
        comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
        correct_comp = join_components(comp_clean_list).title().replace(" And ", " and ")
        
        options_text, correct_letter = build_simple_mcq(correct_comp, list_all_compositions)
        q_text = f"###Question: What are the active ingredients in the medicine {name}?\n###Options:\n{options_text}"
        dataset_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter, "topic": "Composition_Identification"})

    # --- PILLAR 2: Disease -> Active Substances ---
    for use, comps_set in use_to_compositions.items():
        correct_list = sorted(list(comps_set))
        options_text, correct_letter, count_filtered, used_fallback = build_hybrid_mcq(
            correct_list, list_all_compositions, current_use=use, item_to_uses=comp_to_all_uses
        )
        total_semantic_filtered += count_filtered
        if used_fallback: total_fallbacks += 1
            
        q_text = f"###Question: Which of the following therapeutic options is more indicated for {use}?\n###Options:\n{options_text}"
        dataset_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter, "topic": "Therapeutic_Indications"})

    # --- PILLAR 3: Composition -> Brands ---
    for comp_str, brands_set in comp_to_brands.items():
        correct_list = sorted(list(brands_set))
        options_text, correct_letter, _, used_fallback = build_hybrid_mcq(correct_list, list_all_brands)
        if used_fallback: total_fallbacks += 1
            
        q_text = f"###Question: Which of the following brand-name medicines are formulated exclusively with {comp_str}?\n###Options:\n{options_text}"
        dataset_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter, "topic": "Brand_Name_Formulation"})

    # --- PILLAR 4: Active Substance -> MoA ---
    for ingredient, correct_moa_masked in masked_moa_registry.items():
        options_text, correct_letter = build_simple_mcq(correct_moa_masked, list_all_moas)
        q_text = f"###Question: What is the mechanism of action of {ingredient}?\n###Options:\n{options_text}"
        dataset_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter, "topic": "Mechanism_of_Action"})

    print(f"\n[REPORT] Total potential 'correct distractors' filtered out: {total_semantic_filtered}")
    print(f"[REPORT] Times fallback (pool too small) was triggered: {total_fallbacks}")
    print(f"[REPORT] MoAs masked using the Generic Name: {stats_masked_generic}")
    print(f"[REPORT] MoAs masked using a Synonym: {stats_masked_synonym}")
    print(f"[REPORT] Unique MoA texts successfully blinded: {stats_unique_texts_masked} out of {len(unique_moa_registry)}")

    random.shuffle(dataset_mcq)
    return dataset_mcq

if __name__ == "__main__":
    print("--- Generating CDSS Medicines TEST Dataset ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    synonyms_dict = {}
    SYNONYMS_FILE = "../../Data/DrugBank/drug_data_test_status.json" 
    
    if os.path.exists(SYNONYMS_FILE):
        with open(SYNONYMS_FILE, 'r', encoding='utf-8') as f:
            syn_data = json.load(f)
            for item in syn_data:
                gen_name = item.get("generic_name", "").strip()
                syns_str = item.get("sinonimos", "")
                
                # Name cluster (Generic + Synonyms)
                cluster = [gen_name]
                if syns_str:
                    cluster.extend([s.strip() for s in syns_str.split(', ') if s.strip()])
                    
                cluster = list(set([c for c in cluster if c]))
                full_syns_string = ", ".join(cluster)
                
                # Map every term in cluster to complete string
                for term in cluster:
                    synonyms_dict[term.title()] = full_syns_string
    else:
        print(f" [WARNING] Synonyms file {SYNONYMS_FILE} not found. Masking will only use generic names.")

    if os.path.exists(INPUT_FILE):
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        qa_mcq = generate_mcq_dataset(raw_data, synonyms_dict)
        
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(qa_mcq, f, indent=4, ensure_ascii=False)
        print(f"\n -> Generated {len(qa_mcq)} questions successfully.")

--- Generating CDSS Medicines TEST Dataset ---

[REPORT] Total potential 'correct distractors' filtered out: 1145
[REPORT] Times fallback (pool too small) was triggered: 0
[REPORT] MoAs masked using the Generic Name: 401
[REPORT] MoAs masked using a Synonym: 49
[REPORT] Unique MoA texts successfully blinded: 430 out of 459

 -> Generated 3885 questions successfully.


##### **Analysis MoA**

In [ ]:
import json
import os
import re

MEDS_FILE = os.path.join("datasets_split", "dataset_meds_TEST_v2.json")
SYNONYMS_FILE = "../../Data/DrugBank/drug_data_test_status.json" 
OUTPUT_FILE = "intermediate_data/moa_masking_analysis.json"

# --- HELPER ---
def mask_drug_names(text, generic_name, synonyms_string):
    """
    Replaces the generic name AND synonyms with 'this drug' in the text.
    """
    if not text: 
        return text
        
    # 1. Prepare the list of terms to mask
    terms_to_mask = [generic_name.strip()]
    
    if synonyms_string:
        # Split by comma followed by a space to preserve IUPAC names which can include numbers
        raw_syns = [s.strip() for s in synonyms_string.split(', ')]
        
        for s in raw_syns:
            # Ignore tiny fragments or pure numbers, but keep short acronyms (e.g., AZT)
            if len(s) < 2 or s.isnumeric():
                continue
            if s.startswith('-'):
                continue
            terms_to_mask.append(s)
        
    # 2. Remove duplicates and sort by length (longest to shortest)
    terms_to_mask = sorted(list(set([t for t in terms_to_mask if t])), key=len, reverse=True)
    
    masked_text = text
    for term in terms_to_mask:
        pattern = re.compile(rf'\b{re.escape(term)}\b', re.IGNORECASE)
        
        if pattern.search(masked_text):
            def smart_replace(match):
                if match.group()[0].isupper():
                    return "This drug"
                return "this drug"
                
            masked_text = pattern.sub(smart_replace, masked_text)
            
    return masked_text

# --- MAIN GENERATION ---
def generate_moa_analysis():
    print("--- [START] Generating MoA Masking Analysis Report ---")

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    # Load files
    if not os.path.exists(MEDS_FILE) or not os.path.exists(SYNONYMS_FILE):
        print(f"[ERROR] Missing files. Ensure {MEDS_FILE} and {SYNONYMS_FILE} exist.")
        return

    with open(MEDS_FILE, 'r', encoding='utf-8') as f:
        meds_data = json.load(f)
        
    with open(SYNONYMS_FILE, 'r', encoding='utf-8') as f:
        syn_data = json.load(f)

    # Build synonyms dictionary
    synonyms_dict = {}
    for item in syn_data:
        gen_name = item.get("generic_name", "").strip()
        syns_str = item.get("sinonimos", "")
        
        # Create a cluster of all related names for this drug
        cluster = [gen_name]
        if syns_str:
            cluster.extend([s.strip() for s in syns_str.split(', ') if s.strip()])
            
        # Clean and deduplicate the cluster
        cluster = list(set([c for c in cluster if c]))
        
        # Convert the full cluster back into a string to pass to our masking function
        full_syns_string = ", ".join(cluster)
        
        # Map EVERY term in the cluster (title cased) to the full string of all names
        for term in cluster:
            synonyms_dict[term.title()] = full_syns_string

    # 3. Extract unique MoAs
    unique_moa_registry = {}
    for item in meds_data:
        moa_dict = item.get('Mechanism_of_action', {})
        if moa_dict:
            for ing, text in moa_dict.items():
                if text and text.strip():
                    ing_clean = ing.strip().title()
                    if ing_clean not in unique_moa_registry:
                        unique_moa_registry[ing_clean] = text.strip()

    # 4. Build Analysis Data
    analysis_list = []
    total_masked_count = 0  
    
    for ingredient, moa_raw in unique_moa_registry.items():
        synonyms_str = synonyms_dict.get(ingredient, "")
        
        # Apply the masking function
        moa_masked = mask_drug_names(moa_raw, ingredient, synonyms_str)
        
        was_masked = (moa_raw != moa_masked)
        
        if was_masked:
            total_masked_count += 1
            
        drug_report = {
            "drug": ingredient,
            "was_masked": was_masked,  
            "synonyms": synonyms_str,
            "moa_raw": moa_raw,
            "moa_masked": moa_masked
        }
        
        analysis_list.append(drug_report)

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(analysis_list, f, indent=4, ensure_ascii=False)
        
    print(f" -> Success! Generated analysis for {len(analysis_list)} unique MoAs.")
    print(f" -> Total MoAs successfully masked: {total_masked_count} out of {len(analysis_list)}")
    print(f" -> Saved to: {OUTPUT_FILE}\n")

if __name__ == "__main__":
    generate_moa_analysis()
#don't use the drug's own name or a synonym, e.g. Hydrocortisone

--- [START] Generating MoA Masking Analysis Report ---
 -> Success! Generated analysis for 459 unique MoAs.
 -> Total MoAs successfully masked: 430 out of 459
 -> Saved to: intermediate_data/moa_masking_analysis.json



--------------------------------------------------------------------------------------

### **3. Final Aggregation: JSON to JSONL Conversion**
Converts the generated instruction-tuning datasets (Train, Dev, Test Open, and Test MCQ) from standard JSON arrays into JSON Lines (JSONL) format. 

In [36]:
import json
import os

INPUT_DIR = "data_finetuning"
OUTPUT_DIR = os.path.join(INPUT_DIR, "final_datasets")

FILES_TO_CONVERT = [
    "Finetuning_medicines_DEV",
    "Finetuning_medicines_TRAIN",
    "Finetuning_medicines_TEST_mcq",
    "Finetuning_medicines_TEST_open"
]

def convert_json_to_jsonl():
    print("--- Starting JSON to JSONL Conversion ---")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Directory ready: {OUTPUT_DIR}")

    for filename in FILES_TO_CONVERT:
        input_filepath = os.path.join(INPUT_DIR, f"{filename}.json")
        output_filepath = os.path.join(OUTPUT_DIR, f"{filename}.jsonl")
        
        if not os.path.exists(input_filepath):
            print(f"  [WARNING] File not found, skipping: {input_filepath}")
            continue
            
        print(f"  -> Processing: {filename}.json ...")
        
        with open(input_filepath, 'r', encoding='utf-8') as json_file:
            data_list = json.load(json_file)
      
        with open(output_filepath, 'w', encoding='utf-8') as jsonl_file:
            # Iterate through each item in the list and write it as a JSON string on a new line
            for entry in data_list:
                json_string = json.dumps(entry, ensure_ascii=False)
                jsonl_file.write(json_string + '\n')
                
        print(f"     [SUCCESS] Converted and saved to: {output_filepath} ({len(data_list)} lines)")

    print("--- Conversion Complete! ---")

if __name__ == "__main__":
    convert_json_to_jsonl()

--- Starting JSON to JSONL Conversion ---
Directory ready: data_finetuning\final_datasets
  -> Processing: Finetuning_medicines_DEV.json ...
     [SUCCESS] Converted and saved to: data_finetuning\final_datasets\Finetuning_medicines_DEV.jsonl (2313 lines)
  -> Processing: Finetuning_medicines_TRAIN.json ...
     [SUCCESS] Converted and saved to: data_finetuning\final_datasets\Finetuning_medicines_TRAIN.jsonl (10691 lines)
  -> Processing: Finetuning_medicines_TEST_mcq.json ...
     [SUCCESS] Converted and saved to: data_finetuning\final_datasets\Finetuning_medicines_TEST_mcq.jsonl (3885 lines)
  -> Processing: Finetuning_medicines_TEST_open.json ...
     [SUCCESS] Converted and saved to: data_finetuning\final_datasets\Finetuning_medicines_TEST_open.jsonl (3885 lines)
--- Conversion Complete! ---


---

### **4. Few-shot prompting evaluation**

Creation of 5 multiple-choice questions from the training dataset to evaluate baselines, with the same code/reasoning used in the creation of the instruction-tuning dataset for Test.

In [ ]:
import json
import random
import os
import re

random.seed(42)

INPUT_DIR = "datasets_split"      
OUTPUT_DIR = "data_finetuning/final_datasets"    

# Reading from the TRAIN split 
INPUT_FILE = os.path.join(INPUT_DIR, "dataset_meds_TRAIN_v2.json")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "few_shot_examples_meds.jsonl")

# using the test file to know which MoAs are forbidden
TEST_MCQ_FILE = os.path.join(OUTPUT_DIR, "Finetuning_medicines_TEST_mcq.jsonl") # there was repetition in MoA questions, so check which train ones aren't in test before using them for few-shot

INSTRUCTION_MCQ = "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) corresponding to the correct answer."

# --- HELPER ---
def get_forbidden_moas(test_file_path):
    """Scans the Test MCQ dataset and returns a set of forbidden active ingredients for MoA questions."""
    forbidden = set()
    if not os.path.exists(test_file_path):
        print(f" [WARNING] Test file {test_file_path} not found. No MoAs will be blocked!")
        return forbidden
        
    with open(test_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                item = json.loads(line)
                q_text = item.get("input", "").lower()
                
                # Extract the question part ignoring the options
                match = re.search(r'###question:\s*(.*?)\n###options:', q_text, re.DOTALL)
                if match:
                    core_q = match.group(1).strip()
                    # If it's an MoA question, extract the ingredient
                    moa_prefixes = ["what is the mechanism of action of", "explain the mechanism of action of"]
                    for prefix in moa_prefixes:
                        if prefix in core_q:
                            ingredient = core_q.replace(prefix, "").strip()
                            ingredient = re.sub(r'[\?\.!]', '', ingredient).strip()
                            forbidden.add(ingredient)
            except json.JSONDecodeError:
                continue
                
    print(f"[INFO] Loaded {len(forbidden)} forbidden MoA ingredients from Test set to prevent leakage.")
    return forbidden

def join_components(lst):
    """Joins components of a SINGLE formula (e.g., A, B and C)."""
    if not lst: return ""
    if len(lst) == 1: return lst[0]
    return ", ".join(lst[:-1]) + " and " + lst[-1]

def join_options(lst):
    """Joins DIFFERENT therapeutic options or brands (e.g., Option 1 ; Option 2)."""
    if not lst: return ""
    return "; ".join(lst)

def clean_composition_text(text):
    """Removes text inside parentheses and normalizes to lowercase."""
    text = re.sub(r'\s*\(.*?\)', '', text)
    return text.strip().lower()

def mask_drug_names(text, generic_name, synonyms_string):
    """
    Replaces the generic name AND synonyms with 'this drug' in the text.
    """
    if not text: 
        return text, False, False
        
    replaced_generic = False
    replaced_synonym = False
    
    # Prepare the list of terms to mask
    terms_to_mask = [generic_name.strip()]
    
    if synonyms_string:
        # Split by comma followed by a space
        raw_syns = [s.strip() for s in synonyms_string.split(', ')]
        terms_to_mask.extend(raw_syns) # Add the synonyms to the list
        
    # Remove duplicates and sort by length (longest to shortest) to prevent partial word masking
    terms_to_mask = sorted(list(set([t for t in terms_to_mask if t])), key=len, reverse=True)
    
    masked_text = text
    for term in terms_to_mask:
        # Escape the term to safely handle parentheses and brackets in regex
        pattern = re.compile(rf'\b{re.escape(term)}\b', re.IGNORECASE)
        
        # Check if the term exists in the current text
        if pattern.search(masked_text):
            if term.lower() == generic_name.lower():
                replaced_generic = True
            else:
                replaced_synonym = True
                
            def smart_replace(match):
                # If the original word starts with a capital letter, keep it capitalized
                if match.group()[0].isupper():
                    return "This drug"
                return "this drug"
                
            masked_text = pattern.sub(smart_replace, masked_text)
            
    return masked_text, replaced_generic, replaced_synonym

def build_simple_mcq(correct_answer, distractor_pool):
    """Builds a 4-option MCQ given the correct answer and a pool of distractors."""
    valid_distractors = [d for d in distractor_pool if d != correct_answer]
    distractors = random.sample(valid_distractors, min(3, len(valid_distractors)))
    options = [correct_answer] + distractors
    random.shuffle(options)
    
    letters = ['A', 'B', 'C', 'D']
    correct_letter = letters[options.index(correct_answer)]
    options_text = "\n".join([f"{letters[i]}. {opt}" for i, opt in enumerate(options)])
    
    return options_text, correct_letter

def build_hybrid_mcq(correct_items_list, global_pool, current_use=None, item_to_uses=None):
    """
    Builds an MCQ with randomly sized options (1 to 3 items joined by ';').
    Includes a semantic filter to prevent distractors from sharing 
    similar clinical indications with the correct answer.
    """
    valid_distractors = []
    local_filtered_count = 0
    fallback_triggered = False
    
    for d in global_pool:
        if d in correct_items_list:
            continue
            
        # Bidirectional semantic overlap check
        if current_use and item_to_uses:
            distractor_uses = item_to_uses.get(d, set())
            current_lower = current_use.lower().strip()
            has_overlap = False
            
            for d_use in distractor_uses:
                d_use_lower = d_use.lower().strip()
                if (current_lower in d_use_lower) or (d_use_lower in current_lower):
                    has_overlap = True
                    break
                    
            if has_overlap:
                local_filtered_count += 1
                continue 
                
        valid_distractors.append(d)
        
    # 1. Format the correct answer
    if len(correct_items_list) <= 3:
        correct_text = join_options(correct_items_list)
    else:
        correct_text = random.choice(correct_items_list)
        
    # 2. Determine random lengths for the 3 distractor options
    distractor_lengths = [random.randint(1, 3) for _ in range(3)]
    total_needed = sum(distractor_lengths)
    
    # 3. Sample the distractors
    if len(valid_distractors) >= total_needed:
        sampled_distractors = random.sample(valid_distractors, total_needed)
    else:
        fallback_triggered = True
        sampled_distractors = random.choices(valid_distractors, k=total_needed)

    # 4. Construct the 3 wrong options
    distractor_options = []
    start_idx = 0
    for length in distractor_lengths:
        chunk = sampled_distractors[start_idx : start_idx + length]
        distractor_options.append(join_options(chunk))
        start_idx += length
        
    # 5. Assemble and shuffle
    options = [correct_text] + distractor_options
    random.shuffle(options)
    
    letters = ['A', 'B', 'C', 'D']
    correct_letter = letters[options.index(correct_text)]
    options_text = "\n".join([f"{letters[i]}. {opt}" for i, opt in enumerate(options)])
    
    return options_text, correct_letter, local_filtered_count, fallback_triggered

# --- MAIN FUNCTION ---
def generate_mcq_dataset(data, synonyms_dict, forbidden_moas):
    # Initialize separate lists to ensure balanced representation in Few-Shot
    pillar1_mcq = []
    pillar2_mcq = []
    pillar3_mcq = []
    pillar4_mcq = []
    
    pool_compositions = set()
    pool_brands = set()
    use_to_compositions = {}
    comp_to_brands = {}
    unique_moa_registry = {}
    
    comp_to_all_uses = {} 
    
    total_semantic_filtered = 0
    total_fallbacks = 0
    
    stats_masked_generic = 0
    stats_masked_synonym = 0
    stats_unique_texts_masked = 0 
    skipped_leaking_moas = 0

    # --- DATA STRUCTURES ---
    for item in data:
        name = item.get('Medicine Name')
        uses = [u.strip() for u in item.get('Uses', []) if u.strip()]
        comp_raw_list = item.get('Composition', [])
        moa_dict = item.get('Mechanism_of_action', {})

        if not name or not comp_raw_list: continue

        comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
        comp_str = join_components(comp_clean_list).title().replace(" And ", " and ")
        
        pool_compositions.add(comp_str)
        pool_brands.add(name)
        
        if comp_str not in comp_to_all_uses:
            comp_to_all_uses[comp_str] = set()

        for use in uses:
            use_lower = use.lower()
            if use_lower not in use_to_compositions:
                use_to_compositions[use_lower] = set()
            use_to_compositions[use_lower].add(comp_str)
            comp_to_all_uses[comp_str].add(use_lower)

        if comp_str not in comp_to_brands:
            comp_to_brands[comp_str] = set()
        comp_to_brands[comp_str].add(name)

        if moa_dict:
            for ing, text in moa_dict.items():
                if text and text.strip():
                    ing_clean = ing.strip().title()
                    if ing_clean not in unique_moa_registry:
                        unique_moa_registry[ing_clean] = text.strip()

    list_all_compositions = list(pool_compositions)
    list_all_brands = list(pool_brands)

    # --- Pre-mask all MoA ---
    masked_moa_registry = {}
    
    for ingredient, moa_text in unique_moa_registry.items():
        syns_string = synonyms_dict.get(ingredient, "")
        masked_moa, used_gen, used_syn = mask_drug_names(moa_text, ingredient, syns_string)
        
        masked_moa_registry[ingredient] = masked_moa
        
        if used_gen: stats_masked_generic += 1
        if used_syn: stats_masked_synonym += 1
            
        if masked_moa != moa_text:
            stats_unique_texts_masked += 1
    
    list_all_moas = list(masked_moa_registry.values())

    # --- PILLAR 1 ---
    for item in data:
        name = item.get('Medicine Name')
        comp_raw_list = item.get('Composition', [])
        if not name or not comp_raw_list: continue
        
        comp_clean_list = sorted([clean_composition_text(c) for c in comp_raw_list])
        correct_comp = join_components(comp_clean_list).title().replace(" And ", " and ")
        
        options_text, correct_letter = build_simple_mcq(correct_comp, list_all_compositions)
        q_text = f"###Question: What are the active ingredients in the medicine {name}?\n###Options:\n{options_text}"
        pillar1_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter})

    # --- PILLAR 2 ---
    for use, comps_set in use_to_compositions.items():
        correct_list = sorted(list(comps_set))
        options_text, correct_letter, count_filtered, used_fallback = build_hybrid_mcq(
            correct_list, list_all_compositions, current_use=use, item_to_uses=comp_to_all_uses
        )
        total_semantic_filtered += count_filtered
        if used_fallback: total_fallbacks += 1
            
        q_text = f"###Question: Which of the following therapeutic options is more indicated for {use}?\n###Options:\n{options_text}"
        pillar2_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter})

    # --- PILLAR 3 ---
    for comp_str, brands_set in comp_to_brands.items():
        correct_list = sorted(list(brands_set))
        options_text, correct_letter, _, used_fallback = build_hybrid_mcq(correct_list, list_all_brands)
        if used_fallback: total_fallbacks += 1
            
        q_text = f"###Question: Which of the following brand-name medicines are formulated exclusively with {comp_str}?\n###Options:\n{options_text}"
        pillar3_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter})

    # --- PILLAR 4 ---
    for ingredient, correct_moa_masked in masked_moa_registry.items():
        # Skip if ingredient is in the Test set
        if ingredient.lower() in forbidden_moas:
            skipped_leaking_moas += 1
            continue
            
        options_text, correct_letter = build_simple_mcq(correct_moa_masked, list_all_moas)
        q_text = f"###Question: What is the mechanism of action of {ingredient}?\n###Options:\n{options_text}"
        pillar4_mcq.append({"instruction": INSTRUCTION_MCQ, "input": q_text, "output": correct_letter})

    # --- BALANCED FEW-SHOT SELECTION (1 from each pillar + 1 random) ---
    if not (pillar1_mcq and pillar2_mcq and pillar3_mcq and pillar4_mcq):
        print("[ERROR] Not enough questions in all pillars to make a balanced selection.")
        return []

    # Pick exactly one from each pillar
    p1_sample = random.choice(pillar1_mcq)
    p2_sample = random.choice(pillar2_mcq)
    p3_sample = random.choice(pillar3_mcq)
    p4_sample = random.choice(pillar4_mcq)

    # Gather all remaining questions to pick the 5th one
    selected_set = [p1_sample, p2_sample, p3_sample, p4_sample]
    all_remaining = [q for q in (pillar1_mcq + pillar2_mcq + pillar3_mcq + pillar4_mcq) if q not in selected_set]
    
    # Pick the 5th question randomly from what's left
    p5_sample = random.choice(all_remaining)

    # the final, balanced 5-shot list
    final_few_shot_dataset = [p1_sample, p2_sample, p3_sample, p4_sample, p5_sample]
    
    # Shuffle 
    random.shuffle(final_few_shot_dataset)

    print(f"\n[REPORT] Total potential 'correct distractors' filtered out: {total_semantic_filtered}")
    print(f"[REPORT] Times fallback (pool too small) was triggered: {total_fallbacks}")
    print(f"[REPORT] MoAs masked using the Generic Name: {stats_masked_generic}")
    print(f"[REPORT] MoAs masked using a Synonym: {stats_masked_synonym}")
    print(f"[REPORT] Unique MoA texts successfully blinded: {stats_unique_texts_masked} out of {len(unique_moa_registry)}")
    print(f"[REPORT] Data Leakage Guard: Skipped {skipped_leaking_moas} overlapping MoAs!")

    return final_few_shot_dataset

if __name__ == "__main__":
    print("--- Generating Few-Shot Examples (5-Shot) ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 1. Load the forbidden MoAs from the Test Set
    forbidden_moas = get_forbidden_moas(TEST_MCQ_FILE)
    
    synonyms_dict = {}
    SYNONYMS_FILE = "../../Data/DrugBank/drug_data_test_status.json" 
    
    if os.path.exists(SYNONYMS_FILE):
        with open(SYNONYMS_FILE, 'r', encoding='utf-8') as f:
            syn_data = json.load(f)
            for item in syn_data:
                gen_name = item.get("generic_name", "").strip()
                syns_str = item.get("sinonimos", "")
                
                # Name cluster (Generic + Synonyms)
                cluster = [gen_name]
                if syns_str:
                    cluster.extend([s.strip() for s in syns_str.split(', ') if s.strip()])
                    
                cluster = list(set([c for c in cluster if c]))
                full_syns_string = ", ".join(cluster)
                
                # Map every term in cluster to complete string
                for term in cluster:
                    synonyms_dict[term.title()] = full_syns_string
    else:
        print(f" [WARNING] Synonyms file {SYNONYMS_FILE} not found. Masking will only use generic names.")

    if os.path.exists(INPUT_FILE):
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        # Generate the balanced 5 examples, passing the forbidden list
        qa_mcq = generate_mcq_dataset(raw_data, synonyms_dict, forbidden_moas)
        
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            for item in qa_mcq:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
                
        print(f"\n -> Generated {len(qa_mcq)} FEW-SHOT examples successfully in JSONL format.")
    else:
        print(f" [ERROR] Input file {INPUT_FILE} not found!")

--- Generating Few-Shot Examples (5-Shot) ---
[INFO] Loaded 459 forbidden MoA ingredients from Test set to prevent leakage.

[REPORT] Total potential 'correct distractors' filtered out: 2574
[REPORT] Times fallback (pool too small) was triggered: 0
[REPORT] MoAs masked using the Generic Name: 612
[REPORT] MoAs masked using a Synonym: 77
[REPORT] Unique MoA texts successfully blinded: 664 out of 714
[REPORT] Data Leakage Guard: Skipped 410 overlapping MoAs!

 -> Generated 5 FEW-SHOT examples successfully in JSONL format.


### **5.Removing MoA questions from train and dev datasets**

Because we split the dataset by brand name, it is natural for active ingredients and clinical indications to overlap between the training and test sets.  However, this creates a specific data leakage problem for MoA questions. To solve this, we adopted a targeted deduplication strategy: we built an entity-extraction script to isolate the active ingredients specifically from the MoA questions in the Test set. We then cross-referenced this list with the Train and Dev sets, deleting any MoA questions that asked about those exact same ingredients. 


In [38]:
import json
import os
import re

TEST_MCQ_FILE = "data_finetuning/final_datasets/Finetuning_medicines_TEST_mcq.jsonl"
TRAIN_FILE = "data_finetuning/final_datasets/Finetuning_medicines_TRAIN.jsonl"
DEV_FILE = "data_finetuning/final_datasets/Finetuning_medicines_DEV.jsonl"
CLEAN_TRAIN_FILE = "data_finetuning/final_datasets/clean/Finetuning_medicines_TRAIN.jsonl"
CLEAN_DEV_FILE = "data_finetuning/final_datasets/clean/Finetuning_medicines_DEV.jsonl"
DUPLICATES_TRAIN_FILE = "data_finetuning/final_datasets/duplicates/Finetuning_medicines_TRAIN_duplicates.jsonl"
DUPLICATES_DEV_FILE = "data_finetuning/final_datasets/duplicates/Finetuning_medicines_DEV_duplicates.jsonl"

def extract_question_from_mcq(input_text):
    """ Extracts the core question from the TEST MCQ input, ignoring options and tags. """
    match = re.search(r'###Question:\s*(.*?)\n###Options:', input_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip().lower()
    return input_text.strip().lower()

def extract_moa_entity(text):
    """ 
    Filter: Only extracts the active ingredient if it is a MoA question.
    """
    text = text.lower().strip()
    
    # Only target Pillar 4 (MoA) prompts
    moa_prefixes = [
        "what is the mechanism of action of",
        "explain the mechanism of action of"
    ]
    
    for prefix in moa_prefixes:
        if prefix in text:
            # Strip the prefix to isolate the ingredient
            entity = text.replace(prefix, "")
            # Remove trailing punctuation and whitespace
            entity = re.sub(r'[\?\.!]', '', entity).strip()
            return entity
            
    # If it is NOT an MoA question, return None 
    return None

def clean_train_dev_datasets(target_file, clean_output_file, duplicates_output_file, mcq_moa_set, dataset_name):
    print(f"\n--- Processing {dataset_name} Dataset ---")
    if not os.path.exists(target_file):
        print(f" [ERROR] File not found: {target_file}")
        return

    processed_count = 0
    removed_count = 0

    os.makedirs(os.path.dirname(clean_output_file), exist_ok=True)
    os.makedirs(os.path.dirname(duplicates_output_file), exist_ok=True)

    with open(target_file, 'r', encoding='utf-8') as fin, \
         open(clean_output_file, 'w', encoding='utf-8') as fout_clean, \
         open(duplicates_output_file, 'w', encoding='utf-8') as fout_dupes:
        
        for line in fin:
            if not line.strip(): continue
                
            try:
                item = json.loads(line)
                target_input = item.get("input", "")
                
                # 1. Try to extract an MoA entity
                target_moa = extract_moa_entity(target_input)
                
                # 2. If it's an MoA question AND the ingredient is in the Test set -> DUPLICATE
                is_duplicate = False
                if target_moa and target_moa in mcq_moa_set:
                    is_duplicate = True
                
                if is_duplicate:
                    fout_dupes.write(json.dumps(item, ensure_ascii=False) + "\n")
                    removed_count += 1
                else:
                    fout_clean.write(json.dumps(item, ensure_ascii=False) + "\n")
                    
                processed_count += 1
                
            except json.JSONDecodeError:
                print(" [WARNING] Failed to parse a line as JSON. Skipping.")
                continue

    print(f" -> Scanned {processed_count} items in {dataset_name}.")
    print(f" -> Found and isolated {removed_count} leaking MoA questions.")
    print(f" -> Saved clean dataset to: {clean_output_file}")
    print(f" -> Saved duplicates list to: {duplicates_output_file}")

def run_mcq_deduplication():
    print("=== Data leakage prevention: MoA (TRAIN/DEV vs TEST MCQ) ===")
    
    mcq_moa_set = set()
    
    # 1. Load and isolate all MoA ingredients from Test MCQ
    if os.path.exists(TEST_MCQ_FILE):
        with open(TEST_MCQ_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    item = json.loads(line)
                    q = extract_question_from_mcq(item.get("input", ""))
                    if q: 
                        moa_entity = extract_moa_entity(q)
                        if moa_entity:
                            mcq_moa_set.add(moa_entity)
                except json.JSONDecodeError:
                    continue
        print(f"[INFO] Extracted {len(mcq_moa_set)} unique MoA ingredients from Test MCQ.")
    else:
        print(f"[ERROR] Test MCQ file not found: {TEST_MCQ_FILE}")
        return
                
    # 2. Process TRAIN and DEV files
    clean_train_dev_datasets(TRAIN_FILE, CLEAN_TRAIN_FILE, DUPLICATES_TRAIN_FILE, mcq_moa_set, "TRAIN")
    clean_train_dev_datasets(DEV_FILE, CLEAN_DEV_FILE, DUPLICATES_DEV_FILE, mcq_moa_set, "DEV")
    
    print("\n=== COMPLETE ===")

if __name__ == "__main__":
    run_mcq_deduplication()

=== Data leakage prevention: MoA (TRAIN/DEV vs TEST MCQ) ===
[INFO] Extracted 459 unique MoA ingredients from Test MCQ.

--- Processing TRAIN Dataset ---
 -> Scanned 10691 items in TRAIN.
 -> Found and isolated 410 leaking MoA questions.
 -> Saved clean dataset to: data_finetuning/final_datasets/clean/Finetuning_medicines_TRAIN.jsonl
 -> Saved duplicates list to: data_finetuning/final_datasets/duplicates/Finetuning_medicines_TRAIN_duplicates.jsonl

--- Processing DEV Dataset ---
 -> Scanned 2313 items in DEV.
 -> Found and isolated 276 leaking MoA questions.
 -> Saved clean dataset to: data_finetuning/final_datasets/clean/Finetuning_medicines_DEV.jsonl
 -> Saved duplicates list to: data_finetuning/final_datasets/duplicates/Finetuning_medicines_DEV_duplicates.jsonl

=== COMPLETE ===
